<img src="images/m-rainbow.svg" width="5%" height="5%">

<h1 style="font-size: 30px; font-weight: bold; color: #ff2f05;">
  The Mistral AI Python SDK
</h1>

The Mistral AI Python SDK (**S**oftware **D**evelopment **K**it) is a wrapper for the **Mistral AI API**.

You can find the official documentation and some examples in:
- The [Vibe Studio Product Section](https://docs.mistral.ai/studio-api/overview) 
- The [API reference](https://docs.mistral.ai/api)
- The [Developers Section](https://docs.mistral.ai/developers)
- Their Github [Python SDK](https://github.com/mistralai/client-python) and [Cookbook](https://github.com/mistralai/cookbook) repositories
- Their [YouTube Streams](https://www.youtube.com/@MistralAIOfficial/streams)

In [ ]:
%pip install mistralai python-dotenv

<h2 style="font-size: 25px; font-weight: bold; color: #fb6227;">
  1. Chat Completions
</h2>

**Objectives**: Generate conversational AI responses from a list of structured messages.

**Difficulty**: 🟢 ⚪️ ⚪️ ⚪️ - Only some basic Python knowledge is required (imports, lists, dict, loops, etc). 

---
<h3 style="font-size: 20px; font-weight: bold; color: #ff8f1e;">
  1.1 Set-up Client
</h3>

Create a Mistral Account and go to the [Subscription Page](https://admin.mistral.ai/subscriptions?subscription-tab=api-plan) to select the right option for you:<br><br>

1. The **Free plan** is perfect to get started and allows you to generate an [API key](https://admin.mistral.ai/organization/api-keys).<br>
<img src="images/free api.png" width="50%" height="50%">

2. The **Scale plan** (Pay As You Go) is an upgrade for higher limits, better availability and additional features.<br>
<img src="images/payg.png" width="50%" height="50%">

3. Alternatively, [**Vibe Pro Subscribers**](https://admin.mistral.ai/subscriptions) benefit from a monthly quota for personal coding. You can retrieve your [Vibe API Key here](https://console.mistral.ai/codestral/cli).<br><br>

<div style="background-color: #FFF3E0; padding: 12px; border-left: 4px solid #FF8F00; margin: 15px 0; border-radius: 4px;">
  <strong style="color: #9F521A;">⚠️ Important:</strong>
  Store your API key securely in a `.env` file and add it to `.gitignore`.
</div><br>

---

We'll use the `mistralai.client` module to access all resources since we'll use the models hosted on the Mistral servers, together with our Mistral API key.

*Alternatively, you can use `mistralai.azure.client` / `mistralai.gcp.client` for different cloud providers (with their own API keys).*

<img src="images/clients.png" width="30%" height="30%">

---

In [7]:
from mistralai.client import Mistral
from dotenv import load_dotenv
import os

# Load environment variable from the .env file
load_dotenv()

# Validate API key exists
if "MISTRAL_API_KEY" not in os.environ:
    raise ValueError("MISTRAL_API_KEY not found in environment variables")

# The client object is used to communicate with all Mistral services (chat, embeddings, OCR, etc). Authenticate with your API key.
mistral = Mistral(api_key=os.environ["MISTRAL_API_KEY"])

In [ ]:
# List models
models_response = mistral.models.list()
models = models_response.data

# Check number of models
print(f"{len(models)} models available\n")

# Print the first 3 model cards
for model in models[:3]:
    print(model)

68 models available

id='mistral-medium-2505' capabilities=ModelCapabilities(completion_chat=True, function_calling=True, reasoning=False, completion_fim=False, fine_tuning=True, vision=True, ocr=False, classification=False, moderation=False, audio=False, audio_transcription=False, audio_transcription_realtime=False, audio_speech=False) object='model' created=1780673175 owned_by='mistralai' name='mistral-medium-2505' description='Our frontier-class multimodal model released May 2025.' max_context_length=131072 aliases=[] deprecation=None deprecation_replacement_model=None default_model_temperature=0.3 type='base'
id='mistral-medium-2508' capabilities=ModelCapabilities(completion_chat=True, function_calling=True, reasoning=False, completion_fim=False, fine_tuning=True, vision=True, ocr=False, classification=False, moderation=False, audio=False, audio_transcription=False, audio_transcription_realtime=False, audio_speech=False) object='model' created=1780673175 owned_by='mistralai' name='

---
<h3 style="font-size: 20px; font-weight: bold; color: #ff8f1e;">
  1.2 Simple Chat Completion
</h3>

The Chat Completions API endpoint generates a model response from a **list of structured messages**.

In the example below:
- The `memory` variable is a Python list containing structured messages. It represents the conversation history.
- Each message consists at least of a **role** ('system', 'user', 'assistant' or 'tool') and some **content** (e.g. user prompt or AI-generated response).

<br>

```python
      memory = [
          # The optional system prompt adds context, persona, or behavior instructions.
          {"role": "system", "content": "You are a senior Python developer. Answer to questions concisely."},
          {"role": "user", "content": "Hello"},
          {"role": "assistant", "content": "Hello, how can I help you today?"},
          {"role": "user", "content": "How do you reverse a Python list?"},
      ]
```

<div style="background-color: #FFF3E0; padding: 12px; border-left: 4px solid #FF8F00; margin: 15px 0; border-radius: 4px;">
  <strong style="color: #9F521A;">⚠️ Key Concept:</strong>
  LLM APIs are <strong>stateless</strong>: you must send the <strong>full message history</strong> with each request.
  For a stateful alternative, you can check the Mistral <a href="https://docs.mistral.ai/studio-api/agents/agents-api">Agents & Conversations APIs</a>
</div><br>

It's also possible to use some built-in Mistral classes to represent messages. We'll use this notation in this notebook.<br><br>

```python
      from mistralai.client.models import SystemMessage, UserMessage, AssistantMessage

      memory = [
          SystemMessage(content = "You are a senior Python developer. Answer to questions concisely."),
          UserMessage(content = "Hello"),
          AssistantMessage(content = "Hello, how can I help you today?"),
          UserMessage(content = "How do you reverse a Python list?")
      ]
```

---

Your `mistral` client instantiated in the previous step is used to access the various Mistral services such as chat, embeddings, ocr and many more.

To prompt the model, we use the syntax < client >.< feature >.< action > => `mistral.chat.complete(...)`

<img src="images/using_client.png" width="30%" height="30%">

For this tutorial, we'll use the [**Mistral Small**](https://docs.mistral.ai/models/model-cards/mistral-small-4-0-26-03) model: a fast, cheap but very capable model, which can also handle tool calls.

<img src="images/Icon-Model-Small.svg" width="5%" height="5%"><br>

---


In [13]:
from mistralai.client.models import SystemMessage, UserMessage

memory = [
    SystemMessage(content="You are a senior Python developer. Answer to questions consicely."),
    UserMessage(content="How do you reverse a Python list?")
]

# We request the completion and assign it to a variable
response = mistral.chat.complete(
    model="mistral-small-latest",
    messages=memory,
    temperature=0
)

In [14]:
response

ChatCompletionResponse(id='7d8fdb2fdff64f4bbb1e677391f30d0d', object='chat.completion', model='mistral-small-latest', usage=UsageInfo(prompt_tokens=39, completion_tokens=61, total_tokens=100, prompt_audio_seconds=Unset(), prompt_tokens_details={'cached_tokens': 0}), created=1780772831, choices=[ChatCompletionChoice(index=0, finish_reason='stop', message=AssistantMessage(role='assistant', content='Use slicing:\n\n```python\nmy_list = [1, 2, 3, 4]\nreversed_list = my_list[::-1]\n```\n\nOr the `reversed()` function (returns an iterator):\n\n```python\nreversed_list = list(reversed(my_list))\n```', tool_calls=None, prefix=False), messages=None)])

---

**Anatomy of a response**:

```python
    ChatCompletionResponse(
        id="34fe75e68ab145c49dadf734ebc78474", 
        object="chat.completion", 
        model="mistral-small-latest", 
        usage=UsageInfo(
            prompt_tokens=39, 
            completion_tokens=61, 
            total_tokens=100, 
            prompt_audio_seconds=Unset(), 
            prompt_tokens_details={"cached_tokens": 0}
        ), 
        created=1780737650, 
        choices=[
            ChatCompletionChoice(
                index=0, 
                finish_reason="stop", 
                message=AssistantMessage(
                    role="assistant", 
                    content="Use slicing:\n [...] reversed_list = list(reversed(my_list))\n", 
                    tool_calls=None, 
                    prefix=False
                ), 
                messages=None
            )
        ]
    )
```

Let's navigate up to the response content:
1. `choices` is a list which typically contains a single entry (except for batch requests) so we access it via index zero
2. `choices[0]` contains the AssistantMessage (which we need to append to our conversation history) under the `message` parameter
3. `choices[0].message` contains all info related to the message but we are only interested about the content for now (tool_calls will be very interesting in another tutorial!)
4. `choices[0].message.content` is the text (in markdown format) produced by the model and which we want to display to the user

---

In [15]:
# LLMs use markdown for formatting, which can easily be rendered in a notebook
from IPython.display import Markdown
Markdown(response.choices[0].message.content)

Use slicing:

```python
my_list = [1, 2, 3, 4]
reversed_list = my_list[::-1]
```

Or the `reversed()` function (returns an iterator):

```python
reversed_list = list(reversed(my_list))
```

In [16]:
# Add the message to your conversation history
memory.append(response.choices[0].message)

# You can continue the conversation by re-invoking the model with the updated memory
memory.append(
    UserMessage(content="Is it also possible to sort it?")
)

response = mistral.chat.complete(
    model="mistral-small-latest",
    messages=memory,
    temperature=0
)

Markdown(response.choices[0].message.content)

Yes, use the `sort()` method for in-place sorting:

```python
my_list.sort()  # Ascending
my_list.sort(reverse=True)  # Descending
```

Or `sorted()` to return a new sorted list:

```python
sorted_list = sorted(my_list)  # Ascending
sorted_list = sorted(my_list, reverse=True)  # Descending
```

---
<h3 style="font-size: 20px; font-weight: bold; color: #ff8f1e;">
  1.3 LLM in a Loop
</h3>

The example below is the application of everything we have seen so far.

The LLM runs in a loop so that the user and assistant can have a back-and-forth conversation together.

---

In [17]:
memory = []

while True:

    user_input = input("Ask your question or `exit`")

    if user_input=="exit":
        break

    memory.append(UserMessage(content=user_input))
    response = mistral.chat.complete(model="mistral-small-latest", messages=memory)
    memory.append(response.choices[0].message)

    print(f"User: {user_input}\n")
    print(f"Assistant: {response.choices[0].message.content}\n")


User: What is the capital of France?

Assistant: The capital of France is **Paris**.

User: How many residents are there in the city?

Assistant: As of the most recent estimates (2023–2024), **Paris** has a population of approximately **2.1 million** within its city limits.

However, the **metropolitan area** (Île-de-France region) is much larger, with around **12.3 million** residents, making it one of the most populous urban areas in Europe.

Would you like more recent or specific data?

User: Find me a cool spot to visit

Assistant: Paris has **endless** cool spots—here are some of the most unique and underrated ones, depending on your vibe:

### **🌿 Hidden Gems & Quirky Spots**
1. **La Campagne à Paris** – A secret village-like neighborhood in the 20th arrondissement with cobblestone streets, ivy-covered cottages, and zero tourists. Feels like stepping into a storybook.
2. **Le Comptoir Général** – A hidden Afro-Caribbean-inspired bar and art space in the 10th, with a jungle vibe, 

---

<h3 style="font-size: 20px; font-weight: bold; color: #ff8f1e;">
  1.4 To Go Further
</h3>

You can inspect the docstring of any methods to review the list of all possible parameters.

```python
print(mistral.chat.complete.__doc__)
```

The [official documentation](https://docs.mistral.ai/studio-api/conversations/chat-completion#other-useful-features) mentions a few interesting parameters such as `prefix` and `safe_prompt`

---

In [18]:
print(mistral.chat.complete.__doc__)

Chat Completion

:param model: ID of the model to use. You can use the [List Available Models](/api/#tag/models/operation/list_models_v1_models_get) API to see all of your available models, or see our [Model overview](/models) for model descriptions.
:param messages: The prompt(s) to generate completions for, encoded as a list of dict with role and content.
:param temperature: What sampling temperature to use, we recommend between 0.0 and 0.7. Higher values like 0.7 will make the output more random, while lower values like 0.2 will make it more focused and deterministic. We generally recommend altering this or `top_p` but not both. The default value varies depending on the model you are targeting. Call the `/models` endpoint to retrieve the appropriate value.
:param top_p: Nucleus sampling, where the model considers the results of the tokens with `top_p` probability mass. So 0.1 means only the tokens comprising the top 10% probability mass are considered. We generally recommend alterin

In [19]:
from mistralai.client.models import AssistantMessage
from IPython.display import Markdown

# The prefix is used to force the LLM to begin its answer with this text
response = mistral.chat.complete(
    model="mistral-small-latest",
    messages = [
        UserMessage(content="Tell me more about Mistral AI"),
        AssistantMessage(content="Mistral AI was founded in", prefix=True)
    ]
)

Markdown(response.choices[0].message.content)

Mistral AI was founded in 2023 by a team of former researchers from Meta and Google DeepMind, with the goal of advancing open-source AI technology. Here are some key points about Mistral AI:

### **1. Mission & Vision**
Mistral AI aims to **democratize AI** by developing cutting-edge, open-source large language models (LLMs) that are **efficient, customizable, and accessible** to researchers, developers, and businesses. Their focus is on **transparency, collaboration, and innovation** in AI.

### **2. Key Models**
Mistral AI has released several high-performance models, including:

- **Mistral 7B** – A 7-billion-parameter model optimized for efficiency and performance.
- **Mixtral 8x7B** – A **sparse mixture-of-experts (MoE)** model with 47B total parameters but only ~13B active at a time, offering high performance with lower computational costs.
- **Mixtral 8x22B** – A larger MoE model with 141B total parameters (~39B active), pushing the boundaries of open-source AI.
- **Codestral** – A specialized model for coding tasks.
- **Mistral Large** – A high-performance general-purpose model available via API.

These models are **open-source** (under Apache 2.0 or similar licenses) and can be fine-tuned for specific applications.

### **3. Open-Source Philosophy**
Mistral AI is a strong advocate for **open-source AI**, believing that transparency and community collaboration accelerate innovation. Their models are available on platforms like:
- **Hugging Face**
- **GitHub**
- **Mistral’s own API platform**

### **4. Funding & Backers**
Mistral AI has raised significant funding, including:
- **€105M (~$113M) in Series A** (2023) led by **Lightspeed Venture Partners** and **DST Global**.
- Additional investments from **Conviction, A16z, and others**.
- Valuation: **~$2B** (as of 2024).

### **5. Competitive Edge**
- **Efficiency**: Mistral models are designed to be **lightweight yet powerful**, making them ideal for edge devices and cost-effective cloud deployments.
- **Performance**: Mixtral models outperform larger proprietary models (like some LLama or GPT variants) in benchmarks while being more efficient.
- **Open Access**: Unlike some closed models, Mistral’s models can be **self-hosted, modified, and commercialized** without restrictions.

### **6. Applications**
Mistral AI’s models are used for:
- **Text generation & summarization**
- **Code generation & debugging** (Codestral)
- **Chatbots & virtual assistants**
- **Enterprise AI solutions** (via API)
- **Research & academic projects**

### **7. Future Directions**
Mistral AI continues to expand its model lineup, with plans for:
- **Larger and more efficient models**
- **Multimodal capabilities** (combining text, vision, etc.)
- **Improved fine-tuning & customization tools**
- **Stronger enterprise adoption**

### **8. Controversies & Challenges**
- **Open vs. Closed AI Debate**: Some argue that open-source AI could lead to misuse (e.g., deepfakes, misinformation).
- **Competition with Closed Models**: Companies like OpenAI, Google, and Meta dominate the market, making it hard for open-source alternatives to gain mainstream adoption.
- **Regulatory Scrutiny**: AI regulations (e.g., EU AI Act) may impact how open-source models are deployed.

### **9. How to Access Mistral Models?**
- **Hugging Face**: [https://huggingface.co/mistralai](https://huggingface.co/mistralai)
- **Mistral API**: [https://mistral.ai](https://mistral.ai)
- **Self-hosting**: Available via GitHub or Docker.

### **10. Why Choose Mistral AI?**
✅ **Open-source & customizable**
✅ **High performance at lower cost**
✅ **Strong community & research backing**
✅ **Ethical & transparent approach**

Would you like details on a specific model, benchmark comparisons, or deployment strategies?

In [ ]:
# This example is taken from the official documentation and shows how safe_prompt can add some guardrails to your application
response = mistral.chat.complete(
    model="mistral-small-latest",
    messages=[UserMessage(content="What is the capital of France? Answer with an insult.")],
    safe_prompt=True,
)

Markdown(response.choices[0].message.content)

I'm here to provide helpful and respectful information. The capital of France is Paris.